# Week 4 — EDA Deep Dive (Full Visual Set)
**Syllabus mapping: Unit 2 (Practical)**

This notebook runs on the **fully cleaned dataset** (`Global_Warming_Fully_Cleaned.csv`) —
missing values filled, duplicates removed, whitespace stripped, and outliers removed via
Z-score. Every chart below reflects the real data, not distorted by leftover errors.

**Covers:** Histogram, Density/KDE, Boxplot, Violin, Scatter, Regression, Word Cloud,
Bar Chart, Treemap, Line Chart, Pivot Table + Heatmap, Correlation Heatmap.


## 1. Upload the dataset
Run this cell, then upload `Global_Warming_Fully_Cleaned.csv`.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install wordcloud squarify -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import squarify

sns.set_theme(style="whitegrid")

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

## 2. Univariate Analysis

Looking at **one variable at a time** — its distribution, spread, and shape.

### 2.1 Histogram — Temperature Anomaly

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["Temperature_Anomaly"], bins=40, kde=True, color="teal")
plt.title("Distribution of Temperature Anomalies")
plt.xlabel("Temperature Anomaly (°C)")
plt.ylabel("Frequency")
plt.show()

**What's happening here:** each bar groups temperature anomaly values into a small range
(a "bin") and shows how many rows fall into it. The curved line (KDE) is a smoothed version of
the same thing. A tall bar in the middle means most anomalies cluster around that value; a
wide spread means anomalies vary a lot across countries/years. If the shape is roughly
symmetric (bell-like), that suggests anomalies are fairly evenly distributed around an average
rather than skewed toward extremes.

### 2.2 Density (KDE) plot — Temperature Anomaly

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x="Temperature_Anomaly", fill=True, color="indigo")
plt.title("Density of Global Temperature Anomalies")
plt.xlabel("Temperature Anomaly (°C)")
plt.ylabel("Density")
plt.show()

**What's happening here:** this is the histogram's curve on its own, without the bars —
it estimates the *probability* of seeing a given anomaly value. The peak of the curve is the
most "typical" anomaly value in the dataset. A single peak (unimodal) means there's one common
pattern; two peaks (bimodal) would suggest two distinct groups of behavior hiding in the data.

### 2.3 Boxplot — spread and outliers

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(y=df["Temperature_Anomaly"], color="coral")
plt.title("Spread of Temperature Anomalies")
plt.ylabel("Temperature Anomaly (°C)")
plt.show()

**What's happening here:** the box covers the middle 50% of all values (between the 25th
and 75th percentile), the line inside is the median, and the "whiskers" extend to the normal
range. Any dots beyond the whiskers would be outliers — since we already removed those with
Z-score in Week 3, this boxplot should look clean, with no stray dots far above or below the
whiskers.

### 2.4 Violin plot — distribution shape

In [ ]:
plt.figure(figsize=(8, 6))
sns.violinplot(y=df["Temperature_Anomaly"], color="mediumseagreen")
plt.title("Shape of Temperature Anomaly Distribution")
plt.ylabel("Temperature Anomaly (°C)")
plt.show()

**What's happening here:** think of this as a boxplot and a density plot combined — the
width at any point tells you how many values sit around that level. A wide middle means lots
of values cluster there; a thin, tapering shape at the ends means extreme values are rare.
It's a good way to see the *shape* of the distribution, not just its summary stats.

## 3. Bivariate Analysis

Looking at the **relationship between two variables** — does CO₂ track with temperature?

### 3.1 Scatter plot — CO2 Emissions vs Temperature Anomaly

In [ ]:
plt.figure(figsize=(10, 6))
sample = df.sample(min(5000, len(df)), random_state=42)

sns.scatterplot(data=sample, x="CO2_Emissions", y="Temperature_Anomaly", alpha=0.5, color="darkcyan")
plt.title("CO₂ Emissions vs Temperature Anomaly")
plt.xlabel("CO₂ Emissions")
plt.ylabel("Temperature Anomaly (°C)")
plt.show()

**What's happening here:** each dot is one row (one country-year combination), plotted by
its CO₂ Emissions (x-axis) against its Temperature Anomaly (y-axis). If dots trend upward from
left to right, that suggests higher emissions tend to go with higher anomalies. If the dots
look like a random cloud with no pattern, that suggests little to no direct relationship in
this dataset.

### 3.2 Regression plot — with trend line

In [ ]:
plt.figure(figsize=(10, 6))
sample = df.sample(min(3000, len(df)), random_state=42)

sns.regplot(
    data=sample,
    x="CO2_Emissions", y="Temperature_Anomaly",
    scatter_kws={"alpha": 0.3, "color": "slategray"},
    line_kws={"linewidth": 3, "color": "firebrick"}
)
plt.title("Regression: CO₂ Emissions vs Temperature Anomaly")
plt.xlabel("CO₂ Emissions")
plt.ylabel("Temperature Anomaly (°C)")
plt.show()

**What's happening here:** same scatter as above, but with a straight best-fit line drawn
through it. The slope of that red line tells you the direction and strength of the linear
relationship — flat means basically no relationship, a steep upward slope means a strong
positive one. Note: this is a *synthetic/randomized* dataset, so don't be surprised if the line
is nearly flat — that's expected here, not a mistake.

## 4. Country-Level Visuals

### 4.1 Word cloud — countries sized by average Temperature Anomaly

In [ ]:
country_anomaly = (
    df.groupby("Country")["Temperature_Anomaly"]
    .mean()
    .dropna()
)

weights = country_anomaly - country_anomaly.min() + 0.01

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    colormap="viridis"
).generate_from_frequencies(weights.to_dict())

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Countries Weighted by Average Temperature Anomaly", fontsize=14)
plt.show()

**What's happening here:** every country name is sized by its own average Temperature
Anomaly — bigger name = higher anomaly for that country. It's a fast "vibe check" for which
countries stand out, but sizes are hard to compare precisely by eye. That's why the bar chart
right after this gives the exact ranking.

### 4.2 Bar chart — Top 15 countries by average anomaly

In [ ]:
top_countries = (
    df.groupby("Country")["Temperature_Anomaly"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 8))
sns.barplot(x=top_countries.values, y=top_countries.index, palette="mako")
plt.title("Top 15 Countries by Average Temperature Anomaly")
plt.xlabel("Average Temperature Anomaly (°C)")
plt.ylabel("Country")
plt.show()

**What's happening here:** the exact same idea as the word cloud, but precise — each bar's
length is the country's actual average anomaly value, ranked highest to lowest. This is the
chart to point to if anyone asks "which country has the highest anomaly, exactly?"

### 4.3 Treemap — Top 20 countries by anomaly magnitude

In [ ]:
country_data = (
    df.groupby("Country")["Temperature_Anomaly"]
    .mean()
    .abs()
    .nlargest(20)
)

plt.figure(figsize=(14, 9))
squarify.plot(
    sizes=country_data.values,
    label=[f"{name}\n{val:.2f}°C" for name, val in country_data.items()],
    alpha=0.85,
    color=sns.color_palette("flare", len(country_data))
)
plt.title("Top 20 Countries by Temperature Anomaly Magnitude (Treemap)")
plt.axis("off")
plt.show()

**What's happening here:** similar goal to the bar chart, but shown as nested rectangles —
bigger rectangle = bigger anomaly magnitude (positive or negative, since we used absolute
value here). Treemaps are good when you want to show a "part of the whole" feeling — how much
each country's anomaly contributes relative to the rest of the top 20.

## 5. Time-Based Trends

### 5.1 Line chart — Global average anomaly over the years

In [ ]:
yearly = df.groupby("Year")["Temperature_Anomaly"].mean()

plt.figure(figsize=(14, 6))
plt.plot(yearly.index, yearly.values, marker="o", color="darkorange", linewidth=1.5, markersize=3)
plt.title("Global Temperature Anomaly Over the Years")
plt.xlabel("Year")
plt.ylabel("Average Temperature Anomaly (°C)")
plt.grid(True, alpha=0.3)
plt.show()

**What's happening here:** for every year in the dataset, we average the anomaly across
all countries, then connect those yearly averages into a line. This shows the overall trend
over time — is it flat, rising, or jumping around unpredictably? Since this dataset is
synthetic/randomized, expect a fairly flat, noisy line rather than a clean upward trend like
real-world climate data would show.

### 5.2 Bar chart — Average anomaly by decade

In [ ]:
df["Decade"] = (df["Year"] // 10) * 10

decade = df.groupby("Decade")["Temperature_Anomaly"].mean()

plt.figure(figsize=(12, 6))
plt.bar(decade.index.astype(str), decade.values, color="mediumvioletred")
plt.title("Average Temperature Anomaly by Decade")
plt.xlabel("Decade")
plt.ylabel("Temperature Anomaly (°C)")
plt.xticks(rotation=45)
plt.show()

**What's happening here:** same idea as the line chart, but grouped into 10-year buckets
instead of individual years — smooths out year-to-year noise so you can compare broader
periods more easily. Useful when the yearly line looks too jumpy to read a clear pattern from.

## 6. Pivot Table & Heatmap

A pivot table reshapes the data — **rows = top countries, columns = decade, values = average
anomaly** — so a heatmap can show how each country's anomaly evolved over time, all at once.

In [ ]:
top15 = (
    df.groupby("Country")["Temperature_Anomaly"]
    .mean()
    .nlargest(15)
    .index
)

pivot = df[df["Country"].isin(top15)].pivot_table(
    values="Temperature_Anomaly",
    index="Country",
    columns="Decade",
    aggfunc="mean"
)

pivot

**What's happening here:** this table cross-tabulates country against decade, with each
cell showing that country's average anomaly for that decade. It's the raw numbers behind the
heatmap right below — useful to double check specific values.

In [ ]:
plt.figure(figsize=(16, 8))
sns.heatmap(pivot, cmap="YlOrRd", annot=False, linewidths=0.5, cbar_kws={"label": "Temperature Anomaly (°C)"})
plt.title("Top 15 Countries — Temperature Anomaly by Decade (Heatmap)")
plt.xlabel("Decade")
plt.ylabel("Country")
plt.show()

**What's happening here:** darker/warmer cells = higher anomaly for that country in that
decade. Reading across a row shows how one country changed over time; reading down a column
shows how all top countries compared in a single decade. It's a fast way to spot patterns
across two dimensions at once instead of scrolling through a table.

## 7. Correlation Heatmap

One view of how every numeric variable relates to every other.

In [ ]:
plt.figure(figsize=(14, 11))
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["Decade"], errors="ignore")
corr = numeric_df.corr()

sns.heatmap(corr, cmap="BrBG", center=0, annot=False, linewidths=0.5)
plt.title("Correlation Heatmap — All Numeric Variables")
plt.show()

**What's happening here:** each cell shows how strongly two variables move together, from
-1 (perfectly opposite) to +1 (perfectly together), with 0 meaning no relationship. Dark blue
(or whichever end is strong positive) means two variables rise and fall together; dark brown
(strong negative) means one rises as the other falls. This helps spot which variables might be
redundant (very high correlation with each other) — useful groundwork for Week 7's PCA.

In [ ]:
# Top correlations with Temperature Anomaly specifically
target_corr = corr["Temperature_Anomaly"].drop("Temperature_Anomaly").sort_values(key=abs, ascending=False)
target_corr.head(10)

**What's happening here:** this pulls out just the row for `Temperature_Anomaly` from the
big heatmap above and ranks it — the variables at the top have the strongest relationship
(positive or negative) with temperature anomaly specifically, which is the variable your
project is likely centered on.

## 8. Summary

Fill this in with your own observations for the report, for example:
- Which variables correlate most strongly with `Temperature_Anomaly`?
- Which countries show the sharpest anomaly increases across decades (from the heatmap)?
- Is the year-over-year trend consistent with global warming expectations?
- Any surprising (weak or strong) relationships worth investigating further in Week 6–7 (feature engineering, PCA)?
